# Install Packages 

In [ ]:
# Install and load required packages
if (!requireNamespace("lutz", quietly = TRUE)) install.packages("lutz")
if (!requireNamespace("dplyr", quietly = TRUE)) install.packages("dplyr")
if (!requireNamespace("lubridate", quietly = TRUE)) install.packages("lubridate")
if (!requireNamespace("circular", quietly = TRUE)) install.packages("circular")
if (!requireNamespace("arrow", quietly = TRUE)) install.packages("arrow")

library(stringr)
library(bigrquery)
library(data.table)
library(readr)
library(Hmisc)
library(ggplot2)
library(ggrepel)
library(viridis)
library(ggsci)
#library(miceadds)
library(rms)
library(ggpubr)
library(tidyverse)
library(lubridate)
library(dplyr)
library(lubridate)
library(dplyr)
library(lubridate)
library(lutz)
library(data.table)
library(circular)
library(nanoparquet)
library(arrow)

dataset <- Sys.getenv("WORKSPACE_CDR")
my_bucket <- Sys.getenv("WORKSPACE_BUCKET")

In [ ]:
install.packages("usa")

In [ ]:
install.packages(c("timeDate", "lubridate"))

# Define Helper Functions

In [ ]:
#These packages with version numbers are need to be installed

need_package <- function(pkg,version)
{
    need <- FALSE
    if (!require(pkg,character.only = TRUE))
    {
        need <- TRUE
    } else {
        if (packageVersion(pkg) != version)
        {
            need <- TRUE
        }
    }
    return(need)
}

using_version <- function(pkg,version)
{
   if (need_package(pkg,version))
   {
        devtools::install_version(pkg, version = version,upgrade=FALSE)
   }
}

read_bucket <- function (export_path)
{
    col_types <- NULL
    bind_rows(map(system2("gsutil", args = c("ls", export_path),
        stdout = TRUE, stderr = TRUE), function(csv) {
        message(str_glue("Loading {csv}."))
        chunk <- read_csv(pipe(str_glue("gsutil cat {csv}")),
            col_types = col_types, show_col_types = FALSE)
        if (is.null(col_types)) {
            col_types <- spec(chunk)
        }
        chunk
    }))
}

download_data <- function (query)
{
    tb <- bq_project_query(Sys.getenv("GOOGLE_PROJECT"), query)
    bq_table_download(tb, page_size = 1e+05)
}
using_version("Hmisc","4.8-0")
using_version("rms","6.4-1")
using_version("ggsci","3.0.0")
#using_version("miceadds","3.16-18")
#using_version("miceadds","3.17-44")
using_version("ggpubr","0.5.0")

load_stored_result <- FALSE

# Generate SQL Query
The query will cluster to find sleep onset and offset timing

In [ ]:
# convert zip to lat / long to use lutz to get timezone 
library(usa)
zcs <- usa::zipcodes
zcs <- zcs %>%
  mutate(zip3 = substr(zip, 1, 3))

# Aggregate latitude and longitude by 3-digit ZIP code
centroids <- zcs %>%
  group_by(zip3) %>%
  summarise(
    lat = mean(lat, na.rm = TRUE),
    long = mean(long, na.rm = TRUE)
  )

# Using dplyr syntax
zip_to_timezone <- centroids %>%
  select(zip3, lat, long) %>%
  distinct() %>%
  mutate(timezone = tz_lookup_coords(lat, long, method = "fast")) %>%
  select(zip3, timezone)  # This line keeps only zip3 and timezone columns

generate_zip_timezone_struct <- function(zip_to_timezone) {
  struct_strings <- apply(zip_to_timezone, 1, function(row) {
    sprintf("STRUCT('%s' AS zip_code, '%s' AS timezone)", row['zip3'], row['timezone'])
  })
  
  paste(struct_strings, collapse = ",\n        ")
}


## SQL Query 

In [ ]:

zip_timezone_struct <- generate_zip_timezone_struct(zip_to_timezone)
fitbit_query <- str_glue(
"WITH cohort AS (
    SELECT DISTINCT person_id 
    FROM `{dataset}.sleep_daily_summary`
),
person_zip AS (
    SELECT person_id, value_as_string AS zip
    FROM `{dataset}.observation`
    WHERE person_id IN (SELECT person_id FROM cohort)
      AND observation_source_concept_id = 1585250
),
zip_ses AS (
    SELECT DISTINCT z.*EXCEPT(zip3, zip3_as_string), 
           CAST(zip3 AS STRING) as zip_code
    FROM `{dataset}.zip3_ses_map` z
),
zip_timezone AS (
    SELECT zip_code, timezone
    FROM UNNEST([
        {zip_timezone_struct}
    ])
),
socio_eco_data AS (
    SELECT 
        c.person_id,
        z.zip_code,
        zt.timezone
    FROM cohort c
    LEFT JOIN person_zip p ON c.person_id = p.person_id
    LEFT JOIN zip_ses z ON LEFT(p.zip, 3) = z.zip_code
    LEFT JOIN zip_timezone zt ON z.zip_code = zt.zip_code
),
sleep_data AS (
  SELECT
    sl.person_id,
    sl.start_datetime AS utc_start_datetime,
    DATETIME(TIMESTAMP(sl.start_datetime, IFNULL(sed.timezone, 'UTC'))) AS local_start_datetime,
    sl.level,
    sl.duration_in_min,
    CAST(sl.is_main_sleep AS BOOL) as is_main_sleep,  -- Cast string to boolean
    sds.minute_asleep,
    DATE(TIMESTAMP(sl.sleep_date, IFNULL(sed.timezone, 'UTC'))) AS local_sleep_date,
    sed.timezone
  FROM `{dataset}.sleep_level` sl
  INNER JOIN `{dataset}.sleep_daily_summary` sds
    ON sl.person_id = sds.person_id
    AND DATE(sl.sleep_date) = DATE(sds.sleep_date)
  LEFT JOIN socio_eco_data sed ON sl.person_id = sed.person_id
  WHERE sds.minute_asleep > 90
),
time_diff_calc AS (
  SELECT 
    *,
    CAST(TIMESTAMP_DIFF(
      utc_start_datetime,
      LAG(utc_start_datetime) OVER (PARTITION BY person_id ORDER BY utc_start_datetime),
      SECOND
    ) AS INT64) as time_diff
  FROM sleep_data
),
clustered_data AS (
  SELECT 
    *,
    SUM(
      IF(COALESCE(time_diff, 14401) > 14400, 1, 0)
    ) OVER (PARTITION BY person_id ORDER BY utc_start_datetime) as cluster_id
  FROM time_diff_calc
)
SELECT 
  person_id,
  cluster_id,
  MIN(utc_start_datetime) as cluster_start_utc,
  MAX(utc_start_datetime) as cluster_end_utc,
  MIN(local_start_datetime) as cluster_start_local,
  MAX(local_start_datetime) as cluster_end_local,
  TIMESTAMP_DIFF(MAX(utc_start_datetime), MIN(utc_start_datetime), MINUTE) as cluster_duration_mins,
  COUNT(*) as n_observations,
  STRING_AGG(CAST(level as STRING) ORDER BY utc_start_datetime) as level_sequence,
  MAX(timezone) as timezone,
  MIN(local_sleep_date) as local_sleep_date,
  SUM(duration_in_min) as total_duration_mins,
  AVG(duration_in_min) as avg_duration_mins,
  COUNT(DISTINCT level) as unique_levels,
  SUM(IF(is_main_sleep, 1, 0)) as main_sleep_count
FROM clustered_data
GROUP BY person_id, cluster_id
HAVING 
  cluster_duration_mins BETWEEN 60 AND 720
  AND n_observations >= 3
ORDER BY person_id, cluster_start_utc
")

# Fetch Data

In [ ]:
# Export clustered sleep-level query to GCS shards (Parquet) and consolidate to local Parquet
CDR_ENV_VAR <- Sys.getenv("WORKSPACE_CDR")
BILLING_ENV_VAR <- Sys.getenv("GOOGLE_PROJECT")
AOU_RAW_DIR <- file.path(getwd(), "aou_raw_data")
if (!dir.exists(AOU_RAW_DIR)) dir.create(AOU_RAW_DIR, recursive = TRUE)

base_id <- gsub("\\.", "_", CDR_ENV_VAR)
# table identifier for this export
table_name <- "sleep_level_clusters"

# GCS shards path
my_bucket <- Sys.getenv("WORKSPACE_BUCKET")
gcs_temp_shards <- file.path(my_bucket, "bq_exports", table_name, paste0(table_name, "_*.parquet"))

# 3. Export from BQ to GCS as Parquet
bq_table_save(
  bq_dataset_query(CDR_ENV_VAR, fitbit_query, billing = BILLING_ENV_VAR),
  gcs_temp_shards,
  destination_format = "PARQUET",
  write_disposition = "WRITE_TRUNCATE"
)

# 4. Define final local path
final_local_path <- file.path(AOU_RAW_DIR, paste0(base_id, "_", table_name, ".parquet"))

# 5. Read GCS shards and write to single local Parquet file
message("Consolidating shards to single file: ", final_local_path)

# Get the list of GCS shard paths
gcs_shard_files <- system2('gsutil', args = c('ls', gcs_temp_shards), stdout = TRUE, stderr = TRUE)

if (length(gcs_shard_files) == 0 || grepl("No such file", gcs_shard_files[1])) {
   message(paste("WARNING: No files found for", table_name, "at", gcs_temp_shards))
   message("This may be OK if the table has no data for this cohort.")
} else {
  arrow::open_dataset(gcs_shard_files) %>%
    arrow::write_parquet(final_local_path)
}


In [ ]:
# Load consolidated local parquet instead of re-reading shards via CSV helper
CDR_ENV_VAR <- Sys.getenv("WORKSPACE_CDR")
base_id <- gsub("\\.", "_", CDR_ENV_VAR)
local_parquet_path <- file.path(getwd(), "aou_raw_data", paste0(base_id, "_sleep_level_clusters.parquet"))
if (!file.exists(local_parquet_path)) {
  stop("Expected consolidated parquet not found: ", local_parquet_path,
       "\nRun the export cell above first.")
}
message("Loading consolidated parquet: ", local_parquet_path)
sleep_cluster <- arrow::read_parquet(local_parquet_path)

In [ ]:
head(sleep_cluster)

# Data Test

# Long Daily Level Data Aggregation

In [ ]:
# Helper function to get US holidays
get_us_holidays <- function(years) {
  holidays_list <- c()
  
  for(year in years) {
    # Major US holidays
    year_holidays <- c(
      # New Year's Day
      as.Date(paste(year, "01-01", sep="-")),
      
      # Martin Luther King Jr. Day (3rd Monday in January)
      as.Date(paste(year, "01-01", sep="-")) + days(14 + (2 - wday(as.Date(paste(year, "01-01", sep="-")))) %% 7),
      
      # Presidents' Day (3rd Monday in February)
      as.Date(paste(year, "02-01", sep="-")) + days(14 + (2 - wday(as.Date(paste(year, "02-01", sep="-")))) %% 7),
      
      # Memorial Day (last Monday in May)
      as.Date(paste(year, "06-01", sep="-")) - days(wday(as.Date(paste(year, "06-01", sep="-"))) - 2),
      
      # Independence Day
      as.Date(paste(year, "07-04", sep="-")),
      
      # Labor Day (1st Monday in September)
      as.Date(paste(year, "09-01", sep="-")) + days((2 - wday(as.Date(paste(year, "09-01", sep="-")))) %% 7),
      
      # Columbus Day (2nd Monday in October)
      as.Date(paste(year, "10-01", sep="-")) + days(7 + (2 - wday(as.Date(paste(year, "10-01", sep="-")))) %% 7),
      
      # Veterans Day
      as.Date(paste(year, "11-11", sep="-")),
      
      # Thanksgiving (4th Thursday in November)
      as.Date(paste(year, "11-01", sep="-")) + days(21 + (5 - wday(as.Date(paste(year, "11-01", sep="-")))) %% 7),
      
      # Christmas Day
      as.Date(paste(year, "12-25", sep="-"))
    )
    
    holidays_list <- c(holidays_list, year_holidays)
  }
  
  return(unique(as.Date(holidays_list)))
}


In [ ]:
library(lubridate)

# Enhanced version with weekend and holiday flags - FIXED
daily_sleep_metrics <- sleep_cluster %>%
  # Extract date for grouping
  mutate(sleep_date = as.Date(cluster_start_utc)) %>%
  
  # Add weekend and holiday flags - SIMPLIFIED VERSION
  mutate(
    # Weekend flag (Saturday = 7, Sunday = 1 in wday())
    is_weekend = wday(sleep_date) %in% c(1, 7),  # Sunday and Saturday
    
    # US Holiday flag - simplified approach
    is_holiday = sleep_date %in% get_us_holidays(unique(year(sleep_date))),
    
    # Combined weekend OR holiday flag
    is_weekend_or_holiday = is_weekend | is_holiday,
    
    # Day of week name for reference
    day_of_week = wday(sleep_date, label = TRUE, abbr = FALSE),
    
    # Month for seasonal analysis
    month = month(sleep_date, label = TRUE)
  ) %>%
  
  group_by(person_id, sleep_date) %>%
  summarize(
    # Keep the flags
    is_weekend = first(is_weekend),
    is_holiday = first(is_holiday),
    is_weekend_or_holiday = first(is_weekend_or_holiday),
    day_of_week = first(day_of_week),
    month = first(month),
    
    # Daily sleep timing (using circular stats)
    daily_start_hour = circular_mean_time(hour(cluster_start_utc) + minute(cluster_start_utc)/60),
    daily_end_hour = circular_mean_time(hour(cluster_end_utc) + minute(cluster_end_utc)/60),
    daily_midpoint_hour = circular_mean_time(c(
      hour(cluster_start_utc) + minute(cluster_start_utc)/60,
      hour(cluster_end_utc) + minute(cluster_end_utc)/60
    )),
    
    # Daily variability
    daily_start_sd = circular_sd_time(hour(cluster_start_utc) + minute(cluster_start_utc)/60),
    daily_end_sd = circular_sd_time(hour(cluster_end_utc) + minute(cluster_end_utc)/60),
    
    # Daily sleep quality metrics
    daily_duration_mins = sum(cluster_duration_mins),
    n_episodes_per_day = n(),
    
    .groups = 'drop'
  ) %>%
  
  # Add individual-level summaries (including separate weekend/weekday stats)
  group_by(person_id) %>%
  mutate(
    # Overall person averages
    person_avg_start = circular_mean_time(daily_start_hour),
    person_avg_end = circular_mean_time(daily_end_hour),
    person_avg_midpoint = circular_mean_time(daily_midpoint_hour),
    
    # Separate weekend vs weekday averages
    person_weekday_avg_start = circular_mean_time(daily_start_hour[!is_weekend_or_holiday]),
    person_weekend_avg_start = circular_mean_time(daily_start_hour[is_weekend_or_holiday]),
    
    person_weekday_avg_end = circular_mean_time(daily_end_hour[!is_weekend_or_holiday]),
    person_weekend_avg_end = circular_mean_time(daily_end_hour[is_weekend_or_holiday]),
    
    person_weekday_avg_midpoint = circular_mean_time(daily_midpoint_hour[!is_weekend_or_holiday]),
    person_weekend_avg_midpoint = circular_mean_time(daily_midpoint_hour[is_weekend_or_holiday]),
    
    # Weekend effect (how much sleep timing shifts on weekends)
    person_weekend_delay_start = person_weekend_avg_start - person_weekday_avg_start,
    person_weekend_delay_end = person_weekend_avg_end - person_weekday_avg_end,
    person_weekend_delay_midpoint = person_weekend_avg_midpoint - person_weekday_avg_midpoint,
    
    # Duration differences
    person_weekday_avg_duration = mean(daily_duration_mins[!is_weekend_or_holiday], na.rm = TRUE),
    person_weekend_avg_duration = mean(daily_duration_mins[is_weekend_or_holiday], na.rm = TRUE),
    person_weekend_extra_sleep = person_weekend_avg_duration - person_weekday_avg_duration,
    
    # Consistency metrics
    person_start_consistency = circular_sd_time(daily_start_hour),
    person_end_consistency = circular_sd_time(daily_end_hour),
    person_midpoint_consistency = circular_sd_time(daily_midpoint_hour),
    
    # Other person-level metrics
    person_avg_duration = mean(daily_duration_mins, na.rm = TRUE),
    person_total_days = n(),
    person_weekend_days = sum(is_weekend_or_holiday),
    person_weekday_days = sum(!is_weekend_or_holiday),
    
    # Daily deviations from appropriate baseline (weekday vs weekend)
    daily_start_deviation = ifelse(is_weekend_or_holiday,
                                  abs(daily_start_hour - person_weekend_avg_start),
                                  abs(daily_start_hour - person_weekday_avg_start)),
    
    daily_end_deviation = ifelse(is_weekend_or_holiday,
                                abs(daily_end_hour - person_weekend_avg_end),
                                abs(daily_end_hour - person_weekday_avg_end))
  ) %>%
  ungroup()


In [ ]:
# Ensure output directory exists, then write
out_dir <- file.path(getwd(), "processed_data")
if (!dir.exists(out_dir)) dir.create(out_dir, recursive = TRUE)
write_parquet(daily_sleep_metrics, file.path(out_dir, "daily_sleep_metrics_enhanced.parquet"))

# Aggregate Data 

In [ ]:
# Function to calculate circular mean of hours
circular_mean_time <- function(hours) {
  # Convert hours to radians (multiply by 2π/24)
  radians <- hours * 2 * pi / 24
  # Calculate mean direction
  mean_rad <- mean.circular(circular(radians))
  # Convert back to hours (divide by 2π/24)
  mean_hour <- mean_rad * 24 / (2 * pi)
  # Ensure result is between 0 and 24
  mean_hour <- mean_hour %% 24
  return(mean_hour)
}

# Function to calculate circular standard deviation
circular_sd_time <- function(hours) {
  # Convert hours to radians
  radians <- hours * 2 * pi / 24
  # Calculate circular SD
  sd_rad <- sd.circular(circular(radians))
  # Convert back to hours
  sd_hours <- sd_rad * 24 / (2 * pi)
  return(sd_hours)
}

cluster_summary <- sleep_cluster %>%
  group_by(person_id) %>%
  summarize(
    # Average times using circular statistics
    avg_start_hour = circular_mean_time(hour(cluster_start_utc) + minute(cluster_start_utc)/60),
    avg_end_hour = circular_mean_time(hour(cluster_end_utc) + minute(cluster_end_utc)/60),
    # Midpoint calculated after getting circular means of start and end
    avg_midpoint_hour = circular_mean_time(c(
      avg_start_hour,
      avg_end_hour
    )),
    # Standard deviations using circular statistics
    sd_start_hour = circular_sd_time(hour(cluster_start_utc) + minute(cluster_start_utc)/60),
    sd_end_hour = circular_sd_time(hour(cluster_end_utc) + minute(cluster_end_utc)/60),
    sd_midpoint_hour = circular_sd_time(c(
      hour(cluster_start_utc) + minute(cluster_start_utc)/60,
      hour(cluster_end_utc) + minute(cluster_end_utc)/60
    )/2),
    # Other metrics
    n_clusters = n(),
    avg_duration_mins = mean(cluster_duration_mins)
  )


In [ ]:

# Create long format data for plotting
plot_data <- cluster_summary %>%
  tidyr::pivot_longer(
    cols = c(avg_start_hour, avg_midpoint_hour, avg_end_hour),
    names_to = "metric",
    values_to = "hour"
  ) %>%
  mutate(
    metric = case_when(
      metric == "avg_start_hour" ~ "Sleep Onset",
      metric == "avg_midpoint_hour" ~ "Sleep Midpoint",
      metric == "avg_end_hour" ~ "Sleep Offset"
    )
  )

# Create combined density plot
ggplot(plot_data, aes(x = hour, fill = metric)) +
  geom_density(alpha = 0.3) +
  scale_x_continuous(breaks = seq(0, 24, 2)) +
  scale_fill_manual(values = c("Sleep Onset" = "blue", 
                              "Sleep Midpoint" = "purple", 
                              "Sleep Offset" = "red")) +
  labs(title = "Distribution of Sleep Timing Parameters (UTC)",
       subtitle = "Average Sleep Onset, Midpoint, and Offset",
       x = "Hour of Day Local",
       y = "Density",
       fill = "Sleep Parameter") +
  theme_minimal() +
  theme(
    legend.position = "bottom",
    plot.title = element_text(hjust = 0.5),
    plot.subtitle = element_text(hjust = 0.5),
    panel.grid.minor = element_blank()
  )

In [ ]:
circular_mean_time(cluster_summary$avg_midpoint_hour)

# Export the Data

In [ ]:
head(cluster_summary)

In [ ]:
# Ensure output directory exists, then write summary timings parquet
out_dir <- file.path(getwd(), "processed_data")
if (!dir.exists(out_dir)) dir.create(out_dir, recursive = TRUE)
write_parquet(cluster_summary, file.path(out_dir, "sleep_timings.parquet"))